# 18 — Reinforcement Learning Foundations

## Goal

Supervised learning provides an explicit target for each prediction.

Reinforcement learning instead studies sequential decisions whose
consequences may appear only after several future interactions.

Before studying policy gradients, actor-critic methods, or PPO, we need
to understand the quantities those algorithms manipulate:

- states and actions,
- environment transitions,
- rewards,
- policies,
- trajectories,
- discounted returns,
- state values,
- action values,
- advantages,
- Bellman relationships,
- and Monte Carlo estimation.

The goal of this lesson is not to memorize equations.

The goal is to understand what each quantity means and why it is needed.

## 1. Agent and Environment

At time step $t$, the agent observes a state

$$
s_t,
$$

selects an action

$$
a_t,
$$

and the environment produces

$$
r_t
$$

and

$$
s_{t+1}.
$$

We will use the convention

$$
s_t
\xrightarrow{a_t}
(r_t,s_{t+1}).
$$

The agent controls the policy.

The environment controls what happens after an action is taken.

## 2. A Tiny Decision Process

We will use the same tiny environment throughout the lesson.

<pre>
                    leave
              ┌────────────→ terminal
              │               reward = 1
              │
            state s0
              │
              │ continue
              ↓
            state s1
              │
       ┌──────┴──────┐
       │             │
      left          right
       ↓             ↓
   terminal       terminal
   reward = 3     reward = 0
</pre>

The environment is deterministic.

The policy may still be stochastic.

In [1]:
type State = str
type Action = str
type Transition = tuple[State, float]
type TrajectoryStep = tuple[
    State,
    Action,
    float,
    State,
]

In [2]:
transitions: dict[
    State,
    dict[Action, Transition],
] = {
    "s0": {
        "leave": (
            "terminal",
            1.0,
        ),
        "continue": (
            "s1",
            0.0,
        ),
    },
    "s1": {
        "left": (
            "terminal",
            3.0,
        ),
        "right": (
            "terminal",
            0.0,
        ),
    },
}

In [3]:
print(transitions["s0"]["continue"])

('s1', 0.0)


In [4]:
policy: dict[State, dict[Action, float]] = {
    "s0": {"leave": 0.5, "continue": 0.5},
    "s1": {"left": 0.25, "right": 0.75},
}

In [5]:
def discounted_returns(rewards: list[float], gamma: float) -> list[float]:
    """Compute the discounted return from every time step.

    Args:
        rewards: Rewards ordered from earliest to latest.
        gamma: Discount factor in the interval [0, 1].

    Returns:
        One discounted return for every reward position.
    """

    if not 0.0 <= gamma <= 1.0:
        raise ValueError("gamma must be between 0 and 1.")

    returns: list[float] = [0.0 for _ in rewards]

    running_return: float = 0.0

    for time_step in reversed(range(len(rewards))):
        running_return = rewards[time_step] + gamma * running_return
        returns[time_step] = running_return

    return returns


In [6]:
rewards: list[float] = [0.0, 3.0]

print(discounted_returns(rewards, gamma=1.0))

print(discounted_returns(rewards, gamma=0.9))

[3.0, 3.0]
[2.7, 3.0]


## 3. From Return to Value

A return

$$
G_t
$$

belongs to one realized trajectory.

If the policy or environment is stochastic, starting from the same state
again may produce a different trajectory and therefore a different return.

A value function asks for the expected return instead.

The state-value function under policy $\pi$ is

$$
V^\pi(s)
=
\mathbb{E}_\pi
\left[
G_t
\mid
s_t=s
\right].
$$

It answers:

> If I am in state $s$ and follow policy $\pi$ from now on, how much
> future discounted reward should I expect on average?

## 4. Action Values

The state-value function averages over the policy's possible actions.

Sometimes we instead want to ask:

> What if I commit to a particular action first?

The action-value function is

$$
Q^\pi(s,a)
=
\mathbb{E}_\pi
\left[
G_t
\mid
s_t=s,\,
a_t=a
\right].
$$

The first action is fixed.

After that action, future actions are again chosen according to policy
$\pi$.

In [7]:
gamma: float = 1.0

v_s1: float = policy["s1"]["left"] * 3.0 + policy["s1"]["right"] * 0.0

q_s0_leave: float = 1.0

q_s0_continue: float = 0.0 + gamma * v_s1

v_s0: float = (
    policy["s0"]["leave"] * q_s0_leave
    + policy["s0"]["continue"] * q_s0_continue
)

print("V(s1):", v_s1)

print("Q(s0, leave):", q_s0_leave)

print("Q(s0, continue):", q_s0_continue)

print("V(s0):", v_s0)

V(s1): 0.75
Q(s0, leave): 1.0
Q(s0, continue): 0.75
V(s0): 0.875


In [8]:
advantage_leave: float = q_s0_leave - v_s0

advantage_continue: float = q_s0_continue - v_s0

print("A(s0, leave):", advantage_leave)

print("A(s0, continue):", advantage_continue)

A(s0, leave): 0.125
A(s0, continue): -0.125


In [9]:
expected_advantage: float = (
    policy["s0"]["leave"] * advantage_leave
    + policy["s0"]["continue"] * advantage_continue
)

print(expected_advantage)

0.0


### Value, Q-Value, and Advantage

$$
V^\pi(s)
$$

asks:

> How good is this state if I follow policy $\pi$?

$$
Q^\pi(s,a)
$$

asks:

> How good is it to take action $a$ here and then follow policy $\pi$?

$$
A^\pi(s,a)
=
Q^\pi(s,a)
-
V^\pi(s)
$$

asks:

> How much better or worse is this action than the policy's usual
> behavior in this state?

## 5. Bellman Relationships

The discounted return has a recursive structure:

$$
G_t
=
r_t
+
\gamma G_{t+1}.
$$

This says:

> The future from now consists of the immediate reward plus the
> discounted future from the next time step.

Bellman equations apply the same idea to expected returns.

### Bellman Expectation Equations

For action values,

$$
Q^\pi(s,a)
=
\mathbb{E}
\left[
r_t
+
\gamma
V^\pi(s_{t+1})
\mid
s_t=s,\,
a_t=a
\right].
$$

For state values,

$$
V^\pi(s)
=
\mathbb{E}_{a\sim\pi(\cdot|s)}
\left[
Q^\pi(s,a)
\right].
$$

Combining them gives

$$
V^\pi(s)
=
\mathbb{E}_{a\sim\pi,\;s'\sim P}
\left[
r
+
\gamma V^\pi(s')
\right].
$$

In [10]:
def bellman_policy_backup(
    state: State,
    values: dict[State, float],
    policy: dict[
        State,
        dict[Action, float],
    ],
    transitions: dict[
        State,
        dict[Action, Transition],
    ],
    gamma: float,
) -> float:
    """Apply one Bellman expectation backup to one state.

    Args:
        state: State whose value should be updated.
        values: Current value estimates.
        policy: Action probabilities under the current policy.
        transitions: Deterministic environment transitions.
        gamma: Discount factor.

    Returns:
        Updated value estimate for the state.
    """
    updated_value: float = 0.0

    for action, action_probability in policy[state].items():
        next_state, reward = transitions[state][action]

        next_value: float = values.get(next_state, 0.0)

        updated_value += action_probability * (reward + gamma * next_value)

    return updated_value

In [11]:
gamma: float = 1.0


values: dict[State, float] = {"s0": 0.0, "s1": 0.0, "terminal": 0.0}

# 1st round
new_values: dict[State, float] = {
    "s0": bellman_policy_backup("s0", values, policy, transitions, gamma),
    "s1": bellman_policy_backup("s1", values, policy, transitions, gamma),
    "terminal": 0.0,
}

print(new_values)

{'s0': 0.5, 's1': 0.75, 'terminal': 0.0}


In [12]:
values = new_values

new_values = {
    "s0": bellman_policy_backup("s0", values, policy, transitions, gamma),
    "s1": bellman_policy_backup("s1", values, policy, transitions, gamma),
    "terminal": 0.0,
}

print(new_values)

{'s0': 0.875, 's1': 0.75, 'terminal': 0.0}


In [13]:
def evaluate_policy(
    policy: dict[
        State,
        dict[Action, float],
    ],
    transitions: dict[
        State,
        dict[Action, Transition],
    ],
    states: list[State],
    gamma: float,
    num_iterations: int,
) -> dict[State, float]:
    """Estimate state values with repeated Bellman backups.

    Args:
        policy: Action probabilities under the current policy.
        transitions: Deterministic environment transitions.
        states: Non-terminal states to evaluate.
        gamma: Discount factor.
        num_iterations: Number of synchronous Bellman backups.

    Returns:
        Estimated value of every state.
    """
    values: dict[State, float] = {state: 0.0 for state in states}

    values["terminal"] = 0.0

    for _ in range(num_iterations):
        new_values: dict[
            State,
            float,
        ] = {"terminal": 0.0}

        for state in states:
            new_values[state] = bellman_policy_backup(
                state, values, policy, transitions, gamma
            )

        values = new_values

    return values

In [14]:
values = evaluate_policy(
    policy=policy,
    transitions=transitions,
    states=["s0", "s1"],
    gamma=1.0,
    num_iterations=10,
)

print(values)

{'terminal': 0.0, 's0': 0.875, 's1': 0.75}


## 6. Monte Carlo Estimation

Value functions are expectations.

For example,

$$
V^\pi(s)
=
\mathbb E_\pi
[
G_t
\mid
s_t=s
].
$$

If the expectation cannot be computed analytically, it can be estimated
from sampled trajectories:

$$
V^\pi(s)
\approx
\frac{1}{N}
\sum_{i=1}^{N}
G^{(i)}.
$$

Monte Carlo methods therefore replace an unknown expectation with an
empirical average of sampled outcomes.

In [15]:
import torch


def sample_action(action_probabilities: dict[Action, float]) -> Action:
    """Sample one action from a discrete policy distribution."""

    threshold: float = torch.rand(()).item()

    cumulative_probability: float = 0.0

    for action, probability in action_probabilities.items():
        cumulative_probability += probability
        if threshold < cumulative_probability:
            return action
    # Protect against tiny floating-point accumulation errors.
    return next(reversed(action_probabilities))

In [16]:
def sample_trajectory(
    start_state: State,
    policy: dict[State, dict[Action, float]],
    transitions: dict[State, dict[Action, Transition]],
) -> list[TrajectoryStep]:
    """Sample one complete trajectory under a policy."""

    trajectory: list[TrajectoryStep] = []

    state: State = start_state

    while state != "terminal":
        action: Action = sample_action(policy[state])
        next_state, reward = transitions[state][action]
        trajectory.append((state, action, reward, next_state))

        state = next_state

    return trajectory

In [17]:
torch.manual_seed(0)

for _ in range(5):
    print(
        sample_trajectory(
            "s0",
            policy,
            transitions,
        )
    )

[('s0', 'leave', 1.0, 'terminal')]
[('s0', 'continue', 0.0, 's1'), ('s1', 'left', 3.0, 'terminal')]
[('s0', 'leave', 1.0, 'terminal')]
[('s0', 'leave', 1.0, 'terminal')]
[('s0', 'continue', 0.0, 's1'), ('s1', 'right', 0.0, 'terminal')]


In [18]:
def trajectory_rewards(trajectory: list[TrajectoryStep]) -> list[float]:
    """Extract rewards from a trajectory."""

    return [
        reward
        for (
            _,
            _,
            reward,
            _,
        ) in trajectory
    ]

In [19]:
trajectory = sample_trajectory("s0", policy, transitions)

rewards = trajectory_rewards(trajectory)

returns = discounted_returns(rewards, gamma=1.0)

print(trajectory)

print(returns)

[('s0', 'continue', 0.0, 's1'), ('s1', 'right', 0.0, 'terminal')]
[0.0, 0.0]


In [20]:
torch.manual_seed(0)

num_episodes: int = 10_000

sampled_returns: list[float] = []

for _ in range(num_episodes):
    trajectory = sample_trajectory("s0", policy, transitions)

    rewards = trajectory_rewards(trajectory)

    returns = discounted_returns(rewards, gamma=1.0)

    sampled_returns.append(returns[0])

monte_carlo_v_s0: float = sum(sampled_returns) / len(sampled_returns)

print("Monte Carlo V(s0):", monte_carlo_v_s0)

print("exact V(s0):", 0.875)

Monte Carlo V(s0): 0.8658
exact V(s0): 0.875


In [21]:
for num_episodes in [
    10,
    100,
    1_000,
    10_000,
]:
    torch.manual_seed(0)

    returns: list[float] = []

    for _ in range(num_episodes):
        trajectory = sample_trajectory("s0", policy, transitions)

        rewards = trajectory_rewards(trajectory)

        episode_returns = discounted_returns(rewards, gamma=1.0)

        returns.append(episode_returns[0])

    estimate: float = sum(returns) / len(returns)

    print(f"{num_episodes:>6}: {estimate:.4f}")

    10: 0.9000
   100: 0.9000
  1000: 0.9040
 10000: 0.8658


## 7. Mapping Reinforcement Learning to Language Models

An autoregressive language model already defines a stochastic policy.

Given a prompt $x$ and previously generated tokens $y_{<t}$, the model
produces

$$
p_\theta(
y_t
\mid
x,y_{<t}
).
$$

In reinforcement-learning notation, this is simply

$$
\pi_\theta(
a_t
\mid
s_t
).
$$

The mapping is

<pre>
RL state
    ↕
prompt + generated prefix

RL action
    ↕
next token

RL policy
    ↕
next-token distribution

RL trajectory
    ↕
generated completion

RL reward
    ↕
scalar feedback on the generated behavior
</pre>

### Sequence-Level Reward

A language-model reward may be assigned only after the complete response
has been generated:

$$
R(x,y).
$$

For example:

<pre>
prompt
    ↓
generate complete response
    ↓
evaluate response
    ↓
scalar reward
</pre>

The reward might come from a human preference signal, a learned reward
model, a verifier, or another task-specific evaluator.

### Probability and Value Are Different Axes

The policy probability

$$
\pi(a\mid s)
$$

describes how likely the agent is to choose an action.

The action value

$$
Q^\pi(s,a)
$$

describes how much future return is expected after choosing that action.

A policy can assign high probability to a low-value action.

Reinforcement learning attempts to change the policy so that probability
mass moves toward higher-value actions.

In [22]:
llm_policy: dict[str, float] = {"Paris": 0.7, "Lyon": 0.3}

q_values: dict[str, float] = {"Paris": 1.0, "Lyon": 0.0}

In [23]:
state_value: float = sum(
    llm_policy[action] * q_values[action] for action in llm_policy
)

print("V(s):", state_value)

V(s): 0.7


In [24]:
advantages: dict[str, float] = {
    action: (q_value - state_value) for action, q_value in q_values.items()
}

print(advantages)

{'Paris': 0.30000000000000004, 'Lyon': -0.7}


In [25]:
expected_advantage: float = sum(
    llm_policy[action] * advantages[action] for action in llm_policy
)

print(expected_advantage)

2.7755575615628914e-17


## 8. Pretraining and RL Use the Same Policy Differently

The language model itself does not need to change fundamentally.

Both pretraining and reinforcement learning use

$$
\pi_\theta(a_t\mid s_t).
$$

The difference is the training signal.

### Pretraining

The dataset provides the target action:

$$
a_t^*.
$$

The loss can directly increase its probability:

$$
-\log
\pi_\theta(
a_t^*
\mid
s_t
).
$$

### Reinforcement learning

There may be no known correct action.

Instead, an action is sampled:

$$
a_t
\sim
\pi_\theta(
\cdot
\mid
s_t
),
$$

and its consequences produce a return or advantage.

The central question becomes:

> How can scalar feedback change the probability of the sampled action?

## 9. Takeaways

Reinforcement learning studies sequential decisions under delayed and
possibly uncertain feedback.

A transition is

$$
s_t
\xrightarrow{a_t}
(r_t,s_{t+1}).
$$

A trajectory contains a sequence of such interactions.

The discounted return is

$$
G_t
=
r_t
+
\gamma r_{t+1}
+
\gamma^2 r_{t+2}
+
\cdots.
$$

The state-value function is the expected return from a state:

$$
V^\pi(s)
=
\mathbb E_\pi[
G_t
\mid
s_t=s
].
$$

The action-value function fixes the first action:

$$
Q^\pi(s,a)
=
\mathbb E_\pi[
G_t
\mid
s_t=s,
a_t=a
].
$$

Advantage measures relative action quality:

$$
A^\pi(s,a)
=
Q^\pi(s,a)
-
V^\pi(s).
$$

Bellman relationships express the recursive structure of future value:

$$
Q^\pi(s,a)
=
\mathbb E[
r_t
+
\gamma
V^\pi(s_{t+1})
].
$$

Monte Carlo estimation replaces an unavailable expectation with sampled
trajectory returns.

For an autoregressive language model:

<pre>
state
    =
prompt + generated prefix

action
    =
next token

policy
    =
next-token distribution

trajectory
    =
generated completion

reward
    =
scalar feedback

value
    =
expected future reward from a prefix

advantage
    =
how much better or worse a token decision is than the policy's
expected behavior from that prefix
</pre>

The language model already provides the policy.

The next problem is learning how reward should change that policy.